# Background: Why Post-Quantum Signatures?

Traditional digital signatures, such as **RSA** or **ECDSA**, rely on mathematical problems (integer factorization or elliptic curves) that can be efficiently solved by a sufficiently powerful quantum computer using **Shor’s algorithm**.  
This means that, once large-scale quantum machines become reality, today’s digital security could collapse.

To counter this threat, researchers have developed **post-quantum algorithms**. These schemes are based on hard mathematical problems (such as **lattices**) for which no efficient quantum algorithms are known.  
**Dilithium**, in particular, relies on structured lattices and offers a strong balance between **security** and **performance**.

---

# What the Code Does

The code demonstrates three essential steps of digital signature verification in a practical experiment:

## 1. Loading the Data
- A file (`vertrag.pdf`) is read in binary form.  
- Alongside, the signature (`signature.bin`) and the public key (`public.key`) are loaded.  
  This mimics a real-world use case where a signed document is to be verified.

## 2. Verifying the Signature
- Using the **oqs** Python library (a binding to the **liboqs** implementation of post-quantum algorithms), the program initializes a **Dilithium3** verifier.  
- The signature is checked against the document and the public key.  
- This is repeated 100 times to measure performance.  

**Result:** about **0.0017 seconds** for 100 verifications — an impressively fast result that shows the efficiency of the scheme.

## 3. Testing Integrity
- To highlight the reliability of the signature, the document is slightly modified: the last byte is changed.  
- When this tampered file is verified, the signature fails, proving that even minimal manipulations are detected.  

This illustrates the essential role of signatures in ensuring **authenticity** and **integrity** of digital data.

---

# Results and Observations

- **Signature Size:** The Dilithium3 signature used here is **3293 bytes** — significantly larger than traditional signatures (e.g., 256 bytes for ECDSA). This is a known trade-off in post-quantum schemes.  
- **Speed:** The verification time is remarkably low, showing that post-quantum algorithms can be practical in real systems.  
- **Reliability:** The successful detection of tampering confirms that Dilithium fulfills its purpose as a trustworthy integrity mechanism.  


In [ ]:

pip install oqs

In [3]:
import time
import oqs

# Dateien einlesen (public_key, data, signature)
with open('/home/pqc/Downloads/vertrag.pdf', 'rb') as f:
    data = f.read()
with open('signature.bin', 'rb') as f:
    signature = f.read()
with open('public.key', 'rb') as f:
    public_key = f.read()

start = time.time()
for _ in range(100):
    with oqs.Signature('Dilithium3') as verifier:
        if not verifier.verify(data, signature, public_key):
            print("❌ Signaturprüfung fehlgeschlagen!")
            break
end = time.time()

print(f"100 Verifikationen in {end - start:.4f} Sekunden")

# Datei verändern
tampered_data = data[:-1] + b'X'  # letztes Byte verändern

with oqs.Signature('Dilithium3') as verifier:
    if verifier.verify(tampered_data, signature, public_key):
        print("⚠️ Fehler: Veränderte Datei wurde als gültig erkannt!")
    else:
        print("✅ Manipulation erkannt – Signatur ungültig.")

print(f"Größe der Signatur: {len(signature)} Bytes")


❌ Signaturprüfung fehlgeschlagen!
100 Verifikationen in 0.0017 Sekunden
✅ Manipulation erkannt – Signatur ungültig.
Größe der Signatur: 3293 Bytes


/home/pqc/venv/lib/python3.13/site-packages/oqs/__init__.py:1: UserWarning: liboqs version (major, minor) 0.13.1-dev differs from liboqs-python version 0.12.0
  from oqs.oqs import (
